# DeepEval 深度評測：金融 RAG 共用基準

與 RAGAS/Phoenix 共用同一份文件與問題集，專注回歸測試與規則化驗收。

## 評測流程圖

```mermaid
flowchart LR
    A["載入金融 PDF 與 Benchmark 題庫"] --> B["建立共用 RAG Baseline"]
    B --> C["產生每題檢索上下文與回答"]
    C --> D["執行工具評測"]
    D --> E["輸出 CSV 供橫向比較"]
```


### Cell 1 說明：安裝 DeepEval 與相依套件

這一格固定 DeepEval 評測環境，避免不同機器版本造成結果不可比。

重點：
- 固定 `deepeval==3.8.4`。
- 安裝 OpenAI SDK 與資料處理依賴。
- 補上 `python-dotenv`，確保 API key 載入一致。


In [1]:
!uv add deepeval==3.8.4 openai pypdf pandas numpy python-dotenv


Resolved 216 packages in 5ms


Audited 214 packages in 26ms


### Cell 2 說明：建立 Vector RAG baseline（text-embedding-3-large + Top-5）

這一格建立與 RAGAS 相同的向量檢索 baseline：

1. PDF 切塊。
2. 用 `text-embedding-3-large` 產生 chunk embeddings。
3. 對 query 做向量檢索（cosine similarity，Top-5）。
4. 生成 baseline 回答，形成 `rag_df`。

補充：
- 使用 embedding cache（JSON）降低重跑成本。


In [2]:
from dotenv import load_dotenv
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI
from pypdf import PdfReader

PROJECT_ROOT = Path(r"/Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian")
load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(PROJECT_ROOT / "lpdd/.env")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Please set OPENAI_API_KEY before running vector RAG baseline.")

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE") or None,
)

PDF_PATH = PROJECT_ROOT / "docs/Benchmark-Governance/data/financial-stability-report-20211108.pdf"
BENCH_PATH = PROJECT_ROOT / "docs/Benchmark-Governance/data/finance-rag-benchmark.json"
RESULT_DIR = PROJECT_ROOT / "docs/Benchmark-Governance/data/results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL = "text-embedding-3-large"
EMBED_CACHE_PATH = RESULT_DIR / f"embedding_cache_{EMBED_MODEL}.json"

assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"
assert BENCH_PATH.exists(), f"Missing benchmark: {BENCH_PATH}"

reader = PdfReader(str(PDF_PATH))
raw_text = "\n".join((p.extract_text() or "") for p in reader.pages)
raw_text = " ".join(raw_text.split())

chunk_size = 1200
stride = 900
chunks = []
for i in range(0, max(len(raw_text) - chunk_size + 1, 1), stride):
    part = raw_text[i : i + chunk_size]
    if len(part) >= 300:
        chunks.append(part)

with open(BENCH_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

if EMBED_CACHE_PATH.exists():
    with open(EMBED_CACHE_PATH, "r", encoding="utf-8") as f:
        embedding_cache = json.load(f)
else:
    embedding_cache = {}


def _text_key(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _normalize_rows(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.clip(norms, 1e-12, None)


def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    keys = [_text_key(t) for t in texts]
    missing = [(k, t) for k, t in zip(keys, texts) if k not in embedding_cache]

    for i in range(0, len(missing), batch_size):
        batch = missing[i : i + batch_size]
        batch_keys = [k for k, _ in batch]
        batch_texts = [t for _, t in batch]

        response = openai_client.embeddings.create(model=EMBED_MODEL, input=batch_texts)
        data_sorted = sorted(response.data, key=lambda x: x.index)
        for k, d in zip(batch_keys, data_sorted):
            embedding_cache[k] = d.embedding

    if missing:
        with open(EMBED_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(embedding_cache, f)

    mat = np.asarray([embedding_cache[k] for k in keys], dtype=np.float32)
    return _normalize_rows(mat)


chunk_embeddings = embed_texts(chunks)


def retrieve(query: str, top_k: int = 5):
    query_vec = embed_texts([query])[0]
    scores = chunk_embeddings @ query_vec
    order = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in order], [float(scores[i]) for i in order]


def generate_baseline_answer(query: str, retrieved_chunks: list[str]) -> str:
    text = " ".join(retrieved_chunks[:2])
    sents = [s.strip() for s in text.replace("?", ".").split(".") if len(s.strip()) > 30]
    selected = sents[:3]
    if not selected:
        return "No grounded answer generated from retrieved context."
    return " ".join(selected)


rows = []
for item in benchmark:
    contexts, scores = retrieve(item["question"], top_k=5)
    answer = generate_baseline_answer(item["question"], contexts)
    rows.append({
        "id": item["id"],
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "topic": item["topic"],
        "retrieved_contexts": contexts,
        "retrieval_scores": scores,
        "answer": answer,
    })

rag_df = pd.DataFrame(rows)
print(f"chunks={len(chunks)}, embed_cache_entries={len(embedding_cache)}")
rag_df.head(3)


chunks=214, embed_cache_entries=222


,id,question,ground_truth,topic,retrieved_contexts,retrieval_scores,answer
0,FSR-Q01,Financial Stability Report 的主要目的為何？,該報告用於呈現聯準會對美國金融系統韌性的評估，並提升透明度與公眾理解。,purpose,[t financial hardship. Monitoring and assessin...,"[0.6142961978912354, 0.6136758923530579, 0.568...",Monitoring and assessing financial stability a...
1,FSR-Q02,報告如何描述金融穩定與聯準會雙重使命的關係？,金融穩定有助於實現充分就業與物價穩定；金融不穩定會干擾信貸流動並造成失業與經濟困難。,purpose,[lable on the Board’s website; see Board of Go...,"[0.5444421768188477, 0.5427120923995972, 0.528...",lable on the Board’s website; see Board of Gov...
2,FSR-Q03,報告框架如何區分 shocks 與 vulnerabilities？,shocks 是難以預測的突發事件；vulnerabilities 是隨時間累積、在壓力情境...,framework,[it provision and payment services. By contras...,"[0.5232314467430115, 0.4914538860321045, 0.491...","it provision and payment services By contrast,..."


### Cell 3 說明：以 DeepEval 進行多維度評測並輸出結果

這一格會：

1. 將每題資料轉成 `LLMTestCase`。
2. 跑五個指標：
   - Answer Relevancy
   - Faithfulness
   - Contextual Precision
   - Hallucination
   - Financial Groundedness（GEval）
3. 轉成 `result_df` 並輸出 CSV。

重要相容點：
- `HallucinationMetric` 需要 `context`，不能只給 `retrieval_context`。


In [3]:
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    FaithfulnessMetric,
    GEval,
    HallucinationMetric,
)
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Please set OPENAI_API_KEY before running DeepEval evaluation.")

test_cases = []
for r in rows:
    test_cases.append(
        LLMTestCase(
            input=r["question"],
            actual_output=r["answer"],
            expected_output=r["ground_truth"],
            retrieval_context=r["retrieved_contexts"],
            context=r["retrieved_contexts"],
            name=r["id"],
        )
    )

metrics = [
    AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini"),
    FaithfulnessMetric(threshold=0.75, model="gpt-4o-mini"),
    ContextualPrecisionMetric(threshold=0.7, model="gpt-4o-mini"),
    HallucinationMetric(threshold=0.75, model="gpt-4o-mini"),
    GEval(
        name="Financial Groundedness",
        evaluation_params=[
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ],
        criteria="Answer should be financially conservative, evidence-grounded, and avoid unsupported claims.",
        threshold=0.75,
        model="gpt-4o-mini",
    ),
]

eval_result = evaluate(test_cases=test_cases, metrics=metrics)

metric_name_map = {
    "Answer Relevancy": "answer_relevancy",
    "Faithfulness": "faithfulness",
    "Contextual Precision": "contextual_precision",
    "Hallucination": "hallucination",
    "Financial Groundedness [GEval]": "financial_groundedness",
    "Financial Groundedness": "financial_groundedness",
}

records = []
for tr in eval_result.test_results:
    row = {
        "tool": "deepeval",
        "id": tr.name,
        "question": tr.input,
        "reference": tr.expected_output,
        "output": tr.actual_output,
        "user_input": tr.input,
        "retrieved_contexts": tr.retrieval_context,
        "response": tr.actual_output,
    }

    for md in tr.metrics_data:
        metric_key = metric_name_map.get(
            md.name,
            md.name.lower().replace(" ", "_").replace("[geval]", "").strip("_"),
        )
        row[f"{metric_key}_score"] = md.score
        row[f"{metric_key}_reason"] = md.reason
        row[f"{metric_key}_success"] = md.success

    records.append(row)

result_df = pd.DataFrame(records)
result_df.to_csv(RESULT_DIR / "deepeval_finance_results.csv", index=False)
result_df.head()


✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Hallucination Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Financial Groundedness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

/Users/caocharles/Library/CloudStorage/OneDrive-個人/GitHub/claude_test/llm-paper-obsidian/.venv/lib/python3.12/sit
e-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Metrics Summary

  - ❌ Answer Relevancy (score: 0.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because the output fails to address the definition of the second type of vulnerability as requested, instead discussing unrelated aspects of cyber risk and events., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.75, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because there are no contradictions present, indicating that the actual output aligns perfectly with the retrieval context., error: None)
  - ❌ Contextual Precision (score: 0.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.00 because all nodes in the retrieval contexts are irrelevant to the input query. Specifically, the first node discusses cyber risk and financial stability but does not define the second type of vulnerability as requested. Similarly, the second node focuses on implications of cyber events, the 

⚠ WARNING: No hyperparameters logged.
» ]8;id=895327;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.78s | token cost: 0.0195324 USD)
» Test Results (8 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 8

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

,tool,id,question,reference,output,user_input,retrieved_contexts,response,answer_relevancy_score,answer_relevancy_reason,...,faithfulness_success,contextual_precision_score,contextual_precision_reason,contextual_precision_success,hallucination_score,hallucination_reason,hallucination_success,financial_groundedness_score,financial_groundedness_reason,financial_groundedness_success
0,deepeval,FSR-Q05,第二類脆弱性在報告中如何定義？,第二類是企業與家戶過度借貸，當收入下滑或資產價值下降時容易造成支出縮減與違約風險。,translates to considering cyber risk to ﬁ nanc...,第二類脆弱性在報告中如何定義？,[translates to considering cyber risk to ﬁ nan...,translates to considering cyber risk to ﬁ nanc...,0.0,The score is 0.00 because the output fails to ...,...,True,0.000000,The score is 0.00 because all nodes in the ret...,False,0.8,The score is 0.80 because while the actual out...,False,0.258453,The Actual Output discusses cyber risk in rela...,False
1,deepeval,FSR-Q07,第四類 funding risk 為何會導致 run 風險？,因部分機構以可短期贖回資金投資較長天期或低流動性資產，壓力下投資人可能集中贖回引發擠兌。,"be forced to cut back lending, sell their asse...",第四類 funding risk 為何會導致 run 風險？,"[be forced to cut back lending, sell their ass...","be forced to cut back lending, sell their asse...",1.0,The score is 1.00 because the response directl...,...,True,0.916667,The score is 0.92 because the relevant nodes a...,True,0.6,The score is 0.60 because while there are some...,True,0.397965,The Actual Output discusses the implications o...,False
2,deepeval,FSR-Q08,報告提到的 CCyB 與壓力測試有何關係？,金融穩定監測所蒐集資訊可協助聯準會設定壓力測試情境，並支援逆週期資本緩衝（CCyB）決策。,ress-test scenarios and decisions regarding th...,報告提到的 CCyB 與壓力測試有何關係？,[ress-test scenarios and decisions regarding t...,ress-test scenarios and decisions regarding th...,1.0,The score is 1.00 because the response directl...,...,True,1.000000,The score is 1.00 because all relevant nodes a...,True,0.0,The score is 0.00 because there are no contrad...,True,0.523445,The Actual Output provides some relevant infor...,False
3,deepeval,FSR-Q03,報告框架如何區分 shocks 與 vulnerabilities？,shocks 是難以預測的突發事件；vulnerabilities 是隨時間累積、在壓力情境...,"it provision and payment services By contrast,...",報告框架如何區分 shocks 與 vulnerabilities？,[it provision and payment services. By contras...,"it provision and payment services By contrast,...",0.8,The score is 0.80 because while the response p...,...,True,1.000000,The score is 1.00 because all relevant nodes a...,True,0.8,The score is 0.80 because while the actual out...,False,0.300000,The Actual Output provides some context about ...,False
4,deepeval,FSR-Q02,報告如何描述金融穩定與聯準會雙重使命的關係？,金融穩定有助於實現充分就業與物價穩定；金融不穩定會干擾信貸流動並造成失業與經濟困難。,lable on the Board’s website; see Board of Gov...,報告如何描述金融穩定與聯準會雙重使命的關係？,[lable on the Board’s website; see Board of Go...,lable on the Board’s website; see Board of Gov...,0.0,The score is 0.00 because the output fails to ...,...,False,1.000000,The score is 1.00 because all relevant nodes a...,True,0.6,The score is 0.60 because while the actual out...,True,0.288187,The Actual Output lacks a clear connection to ...,False


### Cell 4 說明：彙整 DeepEval 整體表現

這一格先看整體平均分。

注意：
- `hallucination_score` 方向通常是 **越低越好**。
- 其餘多數分數通常是 **越高越好**。

因此我們會額外建立 `hallucination_quality = 1 - hallucination_score`，讓方向一致後再做整體品質判讀。


In [4]:
for c in ["hallucination_score", "answer_relevancy_score", "faithfulness_score", "contextual_precision_score", "financial_groundedness_score"]:
    if c in result_df.columns:
        result_df[c] = pd.to_numeric(result_df[c], errors="coerce")

if "hallucination_score" in result_df.columns:
    result_df["hallucination_quality"] = 1 - result_df["hallucination_score"]

summary_cols = [
    c
    for c in [
        "answer_relevancy_score",
        "faithfulness_score",
        "contextual_precision_score",
        "financial_groundedness_score",
        "hallucination_score",
        "hallucination_quality",
    ]
    if c in result_df.columns
]

result_df[summary_cols].mean(numeric_only=True).sort_index()


answer_relevancy_score          0.595833
contextual_precision_score      0.739583
faithfulness_score              0.944444
financial_groundedness_score    0.341735
hallucination_quality           0.375000
hallucination_score             0.625000
dtype: float64

### Cell 5 說明：逐題分數排序與問題聚焦

這一格把每題的關鍵分數整理後排序，快速定位最需要優先修正的題目。


In [5]:
quality_cols = [
    c
    for c in [
        "answer_relevancy_score",
        "faithfulness_score",
        "contextual_precision_score",
        "financial_groundedness_score",
        "hallucination_quality",
    ]
    if c in result_df.columns
]

per_question_df = result_df[["id", "question"] + quality_cols].copy()
per_question_df["overall_quality"] = per_question_df[quality_cols].mean(axis=1, skipna=True)
per_question_df = per_question_df.sort_values("overall_quality", ascending=True).reset_index(drop=True)

display(per_question_df)


,id,question,answer_relevancy_score,faithfulness_score,contextual_precision_score,financial_groundedness_score,hallucination_quality,overall_quality
0,FSR-Q05,第二類脆弱性在報告中如何定義？,0.000000,1.000000,0.000000,0.258453,0.2,0.291691
1,FSR-Q04,報告中的第一類脆弱性是什麼？,0.500000,1.000000,0.000000,0.201765,0.2,0.380353
2,FSR-Q02,報告如何描述金融穩定與聯準會雙重使命的關係？,0.000000,0.666667,1.000000,0.288187,0.4,0.470971
3,FSR-Q06,第三類脆弱性與金融機構行為有何關聯？,0.800000,1.000000,1.000000,0.266919,0.2,0.653384
4,FSR-Q03,報告框架如何區分 shocks 與 vulnerabilities？,0.800000,1.000000,1.000000,0.300000,0.2,0.660000
5,FSR-Q01,Financial Stability Report 的主要目的為何？,0.666667,0.888889,1.000000,0.497143,0.4,0.690540
6,FSR-Q07,第四類 funding risk 為何會導致 run 風險？,1.000000,1.000000,0.916667,0.397965,0.4,0.742926
7,FSR-Q08,報告提到的 CCyB 與壓力測試有何關係？,1.000000,1.000000,1.000000,0.523445,1.0,0.904689


### Cell 6 說明：單一問題評測解析（以 FSR-Q01 為例）

這一格會把 `DeepEval` 的單題評分流程完整拆開，讓新人知道每個分數到底是怎麼來的。

#### 一、欄位映射（Data -> LLMTestCase）

- `question` / `user_input` -> `input`
- `output` / `response` -> `actual_output`
- `reference` -> `expected_output`
- `retrieved_contexts` -> `retrieval_context` 與 `context`

#### 二、單題評測流程圖

```mermaid
flowchart TB
    Q["Question / input"] --> ANS["actual_output"]
    Q --> RET["retrieved_contexts"]
    REF["reference / expected_output"] --> AR["Answer Relevancy"]
    Q --> AR
    ANS --> AR

    Q --> FA["Faithfulness"]
    ANS --> FA
    RET --> FA

    Q --> CP["Contextual Precision"]
    RET --> CP
    REF --> CP

    Q --> HL["Hallucination"]
    ANS --> HL
    RET --> HL

    Q --> FG["Financial Groundedness (GEval)"]
    ANS --> FG
    REF --> FG
```

#### 三、分數方向與判讀

- `answer_relevancy_score`：越高越好（是否切題）。
- `faithfulness_score`：越高越好（回答是否被檢索證據支持）。
- `contextual_precision_score`：越高越好（檢索內容是否精準有用）。
- `hallucination_score`：越低越好（幻覺比例），本 notebook 另外換算 `hallucination_quality = 1 - hallucination_score`。
- `financial_groundedness_score`：越高越好（金融語境下是否保守且有依據）。

這格會輸出三個層次：
1. 該題原始資料。
2. 各 metric 的門檻判定表（Pass/Fail + 理由）。
3. 指標之間的關係診斷（例如「切題但不忠實」）。


In [6]:
target_id = "FSR-Q01"
single_case = result_df.query("id == @target_id").copy()

display(single_case)

if not single_case.empty:
    r = single_case.iloc[0]

    # A. 先看該題的評測原始材料（輸入、參考答案、模型輸出、檢索證據）
    raw_view_df = pd.DataFrame(
        [
            {
                "id": r.get("id"),
                "question": r.get("question"),
                "reference": r.get("reference"),
                "output": r.get("output"),
                "retrieved_contexts": r.get("retrieved_contexts"),
            }
        ]
    )
    display(raw_view_df)

    # B. 建立指標規格：方向、門檻、判讀重點
    metric_specs = [
        {
            "metric": "answer_relevancy",
            "score_col": "answer_relevancy_score",
            "reason_col": "answer_relevancy_reason",
            "success_col": "answer_relevancy_success",
            "direction": "higher_is_better",
            "threshold": 0.70,
            "meaning": "回答是否直接回應問題重點",
        },
        {
            "metric": "faithfulness",
            "score_col": "faithfulness_score",
            "reason_col": "faithfulness_reason",
            "success_col": "faithfulness_success",
            "direction": "higher_is_better",
            "threshold": 0.75,
            "meaning": "回答是否被檢索證據支持",
        },
        {
            "metric": "contextual_precision",
            "score_col": "contextual_precision_score",
            "reason_col": "contextual_precision_reason",
            "success_col": "contextual_precision_success",
            "direction": "higher_is_better",
            "threshold": 0.70,
            "meaning": "檢索到的內容是否精準、不是雜訊",
        },
        {
            "metric": "hallucination",
            "score_col": "hallucination_score",
            "reason_col": "hallucination_reason",
            "success_col": "hallucination_success",
            "direction": "lower_is_better",
            "threshold": 0.75,
            "meaning": "回答是否包含脫離證據的敘述（越低越好）",
        },
        {
            "metric": "financial_groundedness",
            "score_col": "financial_groundedness_score",
            "reason_col": "financial_groundedness_reason",
            "success_col": "financial_groundedness_success",
            "direction": "higher_is_better",
            "threshold": 0.75,
            "meaning": "金融語境下是否保守、可驗證、避免過度推論",
        },
    ]

    metric_rows = []
    score_lookup = {}

    for spec in metric_specs:
        raw_score = r.get(spec["score_col"])
        score = pd.to_numeric(pd.Series([raw_score]), errors="coerce").iloc[0]
        score_lookup[spec["metric"]] = score

        if pd.isna(score):
            pass_fail = "N/A"
            quality_score = np.nan
        else:
            if spec["direction"] == "higher_is_better":
                pass_fail = "PASS" if score >= spec["threshold"] else "FAIL"
                quality_score = float(score)
            else:
                pass_fail = "PASS" if score <= spec["threshold"] else "FAIL"
                quality_score = float(1 - score)

        metric_rows.append(
            {
                "metric": spec["metric"],
                "score": score,
                "direction": spec["direction"],
                "threshold": spec["threshold"],
                "pass_fail_by_threshold": pass_fail,
                "quality_equivalent": quality_score,
                "success_flag": r.get(spec["success_col"]),
                "meaning": spec["meaning"],
                "reason": r.get(spec["reason_col"]),
            }
        )

    metric_explain_df = pd.DataFrame(metric_rows)
    display(metric_explain_df)

    # C. 做「分數關係」診斷：同時看多個分數，判斷主要瓶頸在檢索或生成
    rel_findings = []

    ar = score_lookup.get("answer_relevancy")
    fa = score_lookup.get("faithfulness")
    cp = score_lookup.get("contextual_precision")
    hl = score_lookup.get("hallucination")

    if pd.notna(ar) and pd.notna(fa):
        if ar >= 0.7 and fa < 0.75:
            rel_findings.append(
                {
                    "pattern": "切題但不忠實",
                    "condition": "answer_relevancy 高、faithfulness 低",
                    "interpretation": "回答看起來有回答到問題，但內容未被證據充分支持，優先收斂生成規則與引用機制。",
                }
            )

    if pd.notna(cp) and pd.notna(fa):
        if cp < 0.7 and fa < 0.75:
            rel_findings.append(
                {
                    "pattern": "檢索精準度不足",
                    "condition": "contextual_precision 低、faithfulness 低",
                    "interpretation": "檢索段落雜訊偏多或命中不準，先改善切塊、重排序或查詢改寫。",
                }
            )

    if pd.notna(hl) and pd.notna(fa):
        if hl > 0.75 and fa < 0.75:
            rel_findings.append(
                {
                    "pattern": "高幻覺 + 低忠實",
                    "condition": "hallucination 高、faithfulness 低",
                    "interpretation": "回答含超出證據內容的敘述，需強化『證據不足就回答不知道』與引用約束。",
                }
            )

    if not rel_findings:
        rel_findings.append(
            {
                "pattern": "無明顯衝突",
                "condition": "主要分數方向一致",
                "interpretation": "可優先聚焦最低分指標做單點優化，再觀察是否連帶提升其他分數。",
            }
        )

    relation_df = pd.DataFrame(rel_findings)
    display(relation_df)


,tool,id,question,reference,output,user_input,retrieved_contexts,response,answer_relevancy_score,answer_relevancy_reason,...,contextual_precision_score,contextual_precision_reason,contextual_precision_success,hallucination_score,hallucination_reason,hallucination_success,financial_groundedness_score,financial_groundedness_reason,financial_groundedness_success,hallucination_quality
6,deepeval,FSR-Q01,Financial Stability Report 的主要目的為何？,該報告用於呈現聯準會對美國金融系統韌性的評估，並提升透明度與公眾理解。,Monitoring and assessing financial stability a...,Financial Stability Report 的主要目的為何？,[t financial hardship. Monitoring and assessin...,Monitoring and assessing financial stability a...,0.666667,The score is 0.67 because there are several ir...,...,1.0,The score is 1.00 because all relevant nodes a...,True,0.6,The score is 0.60 because while there are alig...,True,0.497143,The Actual Output provides relevant informatio...,False,0.4


,id,question,reference,output,retrieved_contexts
0,FSR-Q01,Financial Stability Report 的主要目的為何？,該報告用於呈現聯準會對美國金融系統韌性的評估，並提升透明度與公眾理解。,Monitoring and assessing financial stability a...,[t financial hardship. Monitoring and assessin...


,metric,score,direction,threshold,pass_fail_by_threshold,quality_equivalent,success_flag,meaning,reason
0,answer_relevancy,0.666667,higher_is_better,0.70,FAIL,0.666667,False,回答是否直接回應問題重點,The score is 0.67 because there are several ir...
1,faithfulness,0.888889,higher_is_better,0.75,PASS,0.888889,True,回答是否被檢索證據支持,The score is 0.89 because the actual output in...
2,contextual_precision,1.000000,higher_is_better,0.70,PASS,1.000000,True,檢索到的內容是否精準、不是雜訊,The score is 1.00 because all relevant nodes a...
3,hallucination,0.600000,lower_is_better,0.75,PASS,0.400000,True,回答是否包含脫離證據的敘述（越低越好）,The score is 0.60 because while there are alig...
4,financial_groundedness,0.497143,higher_is_better,0.75,FAIL,0.497143,False,金融語境下是否保守、可驗證、避免過度推論,The Actual Output provides relevant informatio...


,pattern,condition,interpretation
0,無明顯衝突,主要分數方向一致,可優先聚焦最低分指標做單點優化，再觀察是否連帶提升其他分數。


### Cell 7 結論：如何把 DeepEval 輸出轉成可執行改善項目

請用以下順序寫本輪結論（可直接貼到 PR / 文件）：

1. **整體分數摘要（Cell 4）**
- 先點出平均分最低的 1-2 個指標。
- 若 `hallucination_score` 偏高，同步提 `hallucination_quality` 方便與其他高分較佳指標對齊。

2. **最低品質題目（Cell 5）**
- 指出 `overall_quality` 最低題目與其分數組合。
- 優先看該題是否同時出現 `faithfulness` 低 + `hallucination` 高。

3. **單題根因（Cell 6）**
- 引用 `reason` 欄位，判斷是：
  - 檢索問題（contextual_precision 低）
  - 生成問題（hallucination 高 / faithfulness 低）
  - 指令問題（answer_relevancy 低）

4. **下一輪改版優先序**
- 先檢索：切塊、top-k、重排序、query rewrite。
- 再生成：只允許引用 evidence、證據不足回答不知道、限制推論語氣。
- 最後再調模型與 prompt 細節。

一句話總結格式：
`本輪 DeepEval 顯示主要瓶頸在 <X>，下一輪先做 <Y>，預期優先改善 <Z> 指標。`
